In [ ]:
import pandas as pd
#Commented out as I'm doing transfer learning below
#train the model

#Using stratified k-fold per abby's suggestion

from sklearn.model_selection import StratifiedGroupKFold
import os
from collections import defaultdict
import numpy as np

# --- CONFIG ---
source_dir = r"E:/Datasets/Dataset_B/OLD/masked-only"
n_splits = 5
random_seed = 42
batch_size=16

# --- PREPARE DATA ---
X = []  # list of file paths
y = []  # class labels
groups = []  # video prefixes

for bird_class in os.listdir(source_dir):
    class_path = os.path.join(source_dir, bird_class)
    if not os.path.isdir(class_path):
        continue

    #basing groups on video filenames...
    #to avoid frames that look too similar causing data leakage
    for fname in os.listdir(class_path):
        if not fname.lower().endswith(('.jpg','.jpeg','.png')):
            continue
        video_prefix = fname.split('_')[0]
        X.append(os.path.join(class_path, fname))
        y.append(bird_class)
        groups.append(video_prefix)

X = np.array(X)
y = np.array(y)
groups = np.array(groups)

# --- GROUPED K-FOLD ---
gkf = StratifiedGroupKFold(n_splits=n_splits)

for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups=groups)):
    filepath = f"E:/Datasets/Dataset_B/masked-split_train_val_by_day/models/saved_model_kfold{fold}.h5"

    #load the pre-trained VGG19 from keras
    vgg16 = VGG16(input_shape=(224,224,3), weights='imagenet', include_top=False)
    
    x = vgg16.layers[-1].output
    
    #add dropout and the fully connected layer
    x = Dropout(0.5)(x)
    x = Flatten()(x)
    x = Dense(256, activation='relu')(x)
    #add a dense layer with a value equal to the number of classes
    predictors = Dense(17, activation='softmax')(x)
    vgg16model = Model(inputs=vgg16.input, outputs=predictors)
    
    vgg16model.compile(loss='categorical_crossentropy',
              optimizer=Adam(lr=1e-5),#define the optimizer and the learning rate
              metrics=['acc'])
    # add a critera to save only if there was an improvement in the model comparing
    # to the previous epoch (in this caset the model is saved if there was a decrease in the loss value)
    checkpoint = ModelCheckpoint(filepath, monitor='val_loss', verbose=1, save_best_only=True, mode='min')
    # stop training if there is no improvement in model for 3 consecutives epochs.
    early_stopping_monitor = EarlyStopping(patience=5)
    
    reduce_lr = ReduceLROnPlateau(
        monitor='val_loss',   # metric to monitor
        factor=0.5,           # multiply LR by this factor
        patience=3,           # wait this many epochs without improvement
        min_lr=1e-6,          # minimum LR
        verbose=1
    )
    
    callbacks_list = [checkpoint, early_stopping_monitor, reduce_lr]

    print(f"Fold {fold+1}:")
    print(f"  Train samples: {len(train_idx)}")
    print(f"  Validation samples: {len(val_idx)}")
    
    #images
    X_train, X_val = X[train_idx], X[val_idx]
    #labels
    y_train, y_val = y[train_idx], y[val_idx]

    train_df = pd.DataFrame({"filename": X_train, "class": y_train})
    val_df   = pd.DataFrame({"filename": X_val, "class": y_val})
    train_generator = train_data.flow_from_dataframe(
        dataframe=train_df,
        x_col="filename",
        y_col="class",
        target_size=(224, 224),
        class_mode="categorical",
        batch_size=batch_size,
        shuffle=True
    )

    val_generator = val_data.flow_from_dataframe(
        dataframe=val_df,
        x_col="filename",
        y_col="class",
        target_size=(224, 224),
        class_mode="categorical",
        batch_size=batch_size,
        shuffle=False
    )

    #get the integer labels from the generator
    y_train = train_generator.classes
    
    #get all class indices
    classes = np.unique(y_train)
    
    #compute class weights
    class_weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
    class_weight_dict = dict(zip(classes, class_weights))
    
    print("Class weights:", class_weight_dict)

    model_history=vgg16model.fit(
            train_generator,
            steps_per_epoch=len(train_idx)//batch_size,#number of pictures in training data set divided by the batch size
            epochs=30,
            validation_data=val_generator,
            class_weight=class_weight_dict,
            validation_steps= len(val_idx)//batch_size,#number of pictures in validation data set divided by the batch size
            callbacks=callbacks_list)